In [ ]:
import pandas as pd
import numpy as np
from getpass import getuser

user = getuser()
base = rf'C:\Users\{user}\Documents\GitHub\tb_football'

## 1. Merge goals datasets

In [ ]:
goals_wc = pd.read_excel(rf'{base}\data\out\wiki\men\fifa\wc\goals_wc_fifa.xlsx')
goals_eu = pd.read_excel(rf'{base}\data\out\wiki\men\uefa\eu\goals_eu_uefa.xlsx')

goals_wc['fifa_rule'] = 1
goals_eu['fifa_rule'] = 0

goals = pd.concat([goals_wc, goals_eu], ignore_index=True)
print(f'goals_wc: {len(goals_wc)} rows, goals_eu: {len(goals_eu)} rows, merged: {len(goals)} rows')

### 1a. elo_favorite and elo_underdog

In [ ]:
goals['elo_favorite'] = goals[['elo_home', 'elo_away']].max(axis=1)
goals['elo_underdog'] = goals[['elo_home', 'elo_away']].min(axis=1)

print(goals[['elo_home', 'elo_away', 'elo_favorite', 'elo_underdog']].head())

### 1b. elo1st, elo2nd, elo3rd, elo4th

For each group (year, stage), collect all four teams' elo values and rank them from highest to lowest.

In [ ]:
def add_group_elo_ranks(df):
    # Build (year, stage, team) -> elo mapping from home and away columns
    home = df[['year', 'stage', 'home_team', 'elo_home']].dropna(subset=['home_team', 'elo_home'])
    home = home.rename(columns={'home_team': 'team', 'elo_home': 'elo'})

    away = df[['year', 'stage', 'away_team', 'elo_away']].dropna(subset=['away_team', 'elo_away'])
    away = away.rename(columns={'away_team': 'team', 'elo_away': 'elo'})

    team_elo = pd.concat([home, away]).drop_duplicates(subset=['year', 'stage', 'team'])

    # Rank teams within each group by elo (highest = 1st)
    team_elo['rank'] = team_elo.groupby(['year', 'stage'])['elo'].rank(
        ascending=False, method='first'
    ).astype(int)

    # Pivot to wide: one row per group with elo1st..elo4th
    group_elos = team_elo.pivot_table(
        index=['year', 'stage'], columns='rank', values='elo', aggfunc='first'
    ).reset_index()
    group_elos.columns = ['year', 'stage'] + [f'elo{i}st' if i == 1 else
                                               f'elo{i}nd' if i == 2 else
                                               f'elo{i}rd' if i == 3 else
                                               f'elo{i}th'
                                               for i in group_elos.columns[2:]]
    # Rename to standard names
    rank_cols = {c: f'elo{j}' for j, c in zip(
        ['1st', '2nd', '3rd', '4th'], group_elos.columns[2:]
    )}
    group_elos = group_elos.rename(columns=rank_cols)

    return df.merge(group_elos, on=['year', 'stage'], how='left')


def add_group_elo_ranks_v2(df):
    """Rank teams by elo within each (year, stage) group, assign elo1st-elo4th."""
    home = df[['year', 'stage', 'home_team', 'elo_home']].dropna(subset=['home_team', 'elo_home'])
    home = home.rename(columns={'home_team': 'team', 'elo_home': 'elo'})

    away = df[['year', 'stage', 'away_team', 'elo_away']].dropna(subset=['away_team', 'elo_away'])
    away = away.rename(columns={'away_team': 'team', 'elo_away': 'elo'})

    team_elo = (pd.concat([home, away])
                .drop_duplicates(subset=['year', 'stage', 'team'])
                .sort_values(['year', 'stage', 'elo'], ascending=[True, True, False]))

    # Build group-level series
    records = []
    for (yr, st), grp in team_elo.groupby(['year', 'stage']):
        elos = grp['elo'].values
        rec = {'year': yr, 'stage': st}
        for i, name in enumerate(['elo1st', 'elo2nd', 'elo3rd', 'elo4th']):
            rec[name] = elos[i] if i < len(elos) else np.nan
        records.append(rec)

    group_elos = pd.DataFrame(records)
    return df.merge(group_elos, on=['year', 'stage'], how='left')


goals = add_group_elo_ranks_v2(goals)
print(goals[['year', 'stage', 'elo1st', 'elo2nd', 'elo3rd', 'elo4th']].drop_duplicates(['year', 'stage']).head(10))

### 1c. Save merged goals dataset

In [ ]:
out_path = rf'{base}\data\out\goals_merged.xlsx'
goals.to_excel(out_path, index=False)
print(f'Saved goals_merged.xlsx: {len(goals)} rows, {len(goals.columns)} columns')
print('Columns:', list(goals.columns))

## 2. Merge mbm datasets

In [ ]:
mbm_wc = pd.read_excel(rf'{base}\data\out\wiki\men\fifa\wc\mbm_wc_fifa.xlsx')
mbm_eu = pd.read_excel(rf'{base}\data\out\wiki\men\uefa\eu\mbm_eu_uefa.xlsx')

mbm_wc['fifa_rule'] = 1
mbm_eu['fifa_rule'] = 0

mbm = pd.concat([mbm_wc, mbm_eu], ignore_index=True)
print(f'mbm_wc: {len(mbm_wc)} rows, mbm_eu: {len(mbm_eu)} rows, merged: {len(mbm)} rows')

In [ ]:
# elo1st-elo4th: same logic — group elo ranking by (year, stage)
# Reuse the team-elo mapping derived from the goals data to avoid re-deriving
home = mbm[['year', 'stage', 'home_team', 'elo_home']].dropna(subset=['home_team', 'elo_home'])
home = home.rename(columns={'home_team': 'team', 'elo_home': 'elo'})
away = mbm[['year', 'stage', 'away_team', 'elo_away']].dropna(subset=['away_team', 'elo_away'])
away = away.rename(columns={'away_team': 'team', 'elo_away': 'elo'})

team_elo_mbm = (pd.concat([home, away])
                .drop_duplicates(subset=['year', 'stage', 'team'])
                .sort_values(['year', 'stage', 'elo'], ascending=[True, True, False]))

records = []
for (yr, st), grp in team_elo_mbm.groupby(['year', 'stage']):
    elos = grp['elo'].values
    rec = {'year': yr, 'stage': st}
    for i, name in enumerate(['elo1st', 'elo2nd', 'elo3rd', 'elo4th']):
        rec[name] = elos[i] if i < len(elos) else np.nan
    records.append(rec)

group_elos_mbm = pd.DataFrame(records)
mbm = mbm.merge(group_elos_mbm, on=['year', 'stage'], how='left')
print(mbm[['year', 'stage', 'elo1st', 'elo2nd', 'elo3rd', 'elo4th']].drop_duplicates(['year', 'stage']).head(10))

In [ ]:
out_path_mbm = rf'{base}\data\out\mbm_merged.xlsx'
mbm.to_excel(out_path_mbm, index=False)
print(f'Saved mbm_merged.xlsx: {len(mbm)} rows, {len(mbm.columns)} columns')

## 3. Quick check

In [ ]:
print('=== goals_merged ===')
print(goals.groupby('fifa_rule')[['qual_changed', 'elo_favorite', 'elo_underdog',
                                   'elo1st', 'elo2nd', 'elo3rd', 'elo4th']].mean().round(1))

print('\n=== mbm_merged ===')
print(mbm.groupby('fifa_rule')[['suspense', 'elo1st', 'elo2nd', 'elo3rd', 'elo4th']].mean().round(1))